# 012 · Train a player ReID model (OSNet) on SportsMOT (Google Colab)

**hoop-reid-tracker**, experiment 012. With the players-only detector (011) the tracker sees the right people. What still fails is **identity**: crossings swap IDs, and 17–21 identities are created for 10 players. That is the job of the **appearance model (ReID)**.

### What ReID does in our tracker
For every detection the tracker computes an *embedding*: a vector that summarises how the player looks. Two crops of the same player should give close vectors (small cosine distance), and two different players far ones. The tracker uses that distance:
* **in every frame** (stage 1): cost = 0.5·appearance + 0.5·motion. In a crossing the motion is ambiguous, so appearance decides;
* **after a disappearance** (memory): a new detection is compared with the gallery of lost players before a new ID is created.

### Why the current one fails
Today the embedding comes from a **ResNet-50 trained on ImageNet**. It learned to tell *objects* apart, not people. For it, two team-mates are "a person in a white shirt": almost the same vector. A ReID model trained on players learns the cues that separate team-mates: jersey number, skin, hair, shoes, body shape, sleeves, knee pads.

### The model: OSNet
OSNet (*Omni-Scale Network*, Zhou et al. 2019) is designed for ReID. Each block looks at the image at several scales at once: small details (a number, a shoe) and the whole silhouette. It has 2.2 M parameters, 11× fewer than ResNet-50, so it is also **faster on the laptop CPU**. Deep-EIoU, the reference tracker on SportsMOT, uses it fine-tuned on SportsMOT crops: **+3.9 HOTA**.

### How it learns (two losses together)
1. **Identity loss (cross-entropy)**: a classifier must say *which* of the ~170 training identities each crop is. To succeed, the embedding has to contain what makes each player unique. The classifier is thrown away after training; only the embedding is kept.
2. **Triplet loss (batch-hard)**: for each crop (*anchor*), take the **farthest crop of the same player** (*hardest positive*) and the **closest crop of another player** (*hardest negative*). The loss pushes them apart until the negative is at least a margin farther than the positive. That is exactly what the tracker needs: a distance it can trust.

For the triplet loss each batch must contain several crops of each player. A **P×K sampler** takes P identities × K crops each (here 16 × 4 = 64). The identities are taken *from the same clip*, so team-mates appear in the same batch and the hardest negatives are team-mates: exactly the confusion we want to fix.

### Data and evaluation (no leakage)
| split | content | role |
|---|---|---|
| `train` | crops of the 15 SportsMOT **train** clips (3 games) | learning |
| `test` | crops of the 15 SportsMOT **val** clips (4 games never seen) | final numbers only |

There is **no validation split and no "best epoch"**. The recipe (60 epochs, Adam 3e-4) is fixed in advance from the literature, so nothing is chosen by looking at the test.

**Test protocol, the tracker's situation:** a *query* crop of a player is searched among the crops of **the same clip** (team-mates included), taken **at least 2 seconds away** (50 frames). Adjacent frames are almost identical, and they would make the task trivial.
* **Rank-1**: how often the closest crop is the same player.
* **mAP**: how high all the crops of that player are ranked on average.

### Before running (on your laptop)
1. `python scripts/build_reid_dataset.py` creates `data/reid_players.zip` (~200–300 MB).
2. Upload it to `MyDrive/hooptrack/`.
3. In Colab: **Runtime → Change runtime type → T4 GPU**, then run all cells.

**Time on a T4:** ~1 min per epoch, so ~1 h in total. A checkpoint is written to Drive every epoch, and if Colab disconnects, re-running the notebook resumes training.

## 1 · GPU and libraries
`torchreid` provides the OSNet architecture and its ImageNet weights (downloaded with `gdown`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install torchreid gdown

## 2 · Configuration
* `EPOCHS`, `LR`: the fixed recipe (Deep-EIoU: 60 epochs, Adam 3e-4). The learning rate falls ×10 at 2/3 of the training (step decay).
* `P`, `K`: identities per batch × crops per identity.
* `MARGIN`: how much farther the negative must be than the positive in the triplet loss, in cosine distance.

In [ ]:
import os
from pathlib import Path

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/hooptrack")
    WORK = Path("/content")
else:
    DRIVE_DIR = Path(os.environ.get("HOOPTRACK_DIR", "data"))
    WORK = Path(os.environ.get("HOOPTRACK_WORK", "/tmp/hooptrack"))

ZIP = DRIVE_DIR / "reid_players.zip"
RUN = DRIVE_DIR / "runs" / "reid_osnet_x1_0"          # checkpoints on Drive (survive a disconnection)

EPOCHS = int(os.environ.get("HOOPTRACK_EPOCHS", 60))
LR = 3e-4
P, K = 16, 4                     # 16 identities x 4 crops = batch of 64
MARGIN = 0.3                     # triplet margin (cosine distance)
LABEL_SMOOTH = 0.1               # softer identity targets: less overconfident, better embeddings
PRETRAINED = os.environ.get("HOOPTRACK_PRETRAINED", "1") == "1"   # ImageNet weights for OSNet
GAP = 50                         # test: gallery crops at least 50 frames (2 s) from the query
SEED = 0

assert ZIP.is_file(), f"{ZIP} not found: upload reid_players.zip to MyDrive/hooptrack/"
RUN.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
print("zip:", ZIP, f"({ZIP.stat().st_size / 1e6:.0f} MB) | run:", RUN)

## 3 · Unpack and index the crops
Each file name is `<identity>__<clip>__<frame>.jpg`. That gives the label, the clip (needed for the P×K sampler and for the test) and the time (needed for the 2-second gap).

In [ ]:
import shutil, zipfile, random
import numpy as np, pandas as pd

DATA = WORK / "reid_players"
if not (DATA / "train").is_dir():
    local = WORK / "reid_players.zip"
    shutil.copy(ZIP, local)
    with zipfile.ZipFile(local) as z:
        z.extractall(WORK)
    local.unlink()

def index(split: str) -> pd.DataFrame:
    rows = []
    for p in sorted((DATA / split).glob("*.jpg")):
        pid, clip, frame = p.stem.split("__")
        rows.append((str(p), int(pid), clip, int(frame)))
    return pd.DataFrame(rows, columns=["path", "pid", "clip", "frame"])

train_df, test_df = index("train"), index("test")
# the classifier needs labels 0..N-1
pid2label = {p: i for i, p in enumerate(sorted(train_df.pid.unique()))}
train_df["label"] = train_df.pid.map(pid2label)
NUM_IDS = len(pid2label)
for name, df in (("train", train_df), ("test", test_df)):
    print(f"{name}: {len(df):6d} crops, {df.pid.nunique():4d} identities, {df["clip"].nunique():2d} clips, "
          f"{len(df) / df.pid.nunique():.0f} crops per identity on average")

### What the model has to separate
Crops of **4 players of the same clip** (one row each). Two team-mates in identical jerseys differ only in details. Those details are what the model must learn.

In [ ]:
import cv2
import matplotlib.pyplot as plt

random.seed(SEED)
clip = train_df["clip"].iloc[0]
pids = list(train_df[train_df["clip"] == clip].pid.value_counts().index[:4])
fig, axes = plt.subplots(4, 8, figsize=(12, 12))
for r, pid in enumerate(pids):
    rows = train_df[train_df.pid == pid].sample(8, random_state=SEED, replace=True).sort_values("frame")
    for c, (_, row) in enumerate(rows.iterrows()):
        axes[r, c].imshow(cv2.imread(row.path)[..., ::-1]); axes[r, c].axis("off")
        axes[r, c].set_title(f"id {pid} f{row.frame}", fontsize=7)
plt.tight_layout(); plt.show()

## 4 · Data pipeline
**Augmentations**, each with a reason:
* horizontal flip: a player mirrored is the same player;
* small random crop (pad 10 px, then crop back), because detector boxes are never perfectly aligned;
* **random erasing**: a random rectangle is blanked out. Players are constantly occluded by other players, and the model must not depend on one single part of the body;
* mild brightness/contrast change. Colour is **not** changed: the jersey colour is a real cue (it separates the teams).

**P×K sampler:** pick a clip, then P identities of that clip (P is reduced if the clip has fewer), then K crops of each.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T

torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # same normalisation as the tracker's extractor

train_tf = T.Compose([
    T.Resize((256, 128)),
    T.RandomHorizontalFlip(),
    T.Pad(10), T.RandomCrop((256, 128)),
    T.ColorJitter(brightness=0.2, contrast=0.15),
    T.ToTensor(), T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.5, value="random"),
])
test_tf = T.Compose([T.Resize((256, 128)), T.ToTensor(), T.Normalize(MEAN, STD)])

class Crops(Dataset):
    def __init__(self, df, tf, labels=True):
        self.paths, self.tf = df.path.tolist(), tf
        self.labels = df.label.tolist() if labels else [0] * len(df)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        from PIL import Image
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.labels[i]

class PKSampler(Sampler):
    # Batches of P identities x K crops, identities drawn from ONE clip (team-mates = hard negatives)
    def __init__(self, df, p, k, batches):
        self.p, self.k, self.batches = p, k, batches
        self.by_pid = df.groupby("label").indices                       # label -> row indices
        self.by_clip = {c: g.label.unique().tolist() for c, g in df.groupby("clip")}
        self.clips = [c for c, ids in self.by_clip.items() if len(ids) >= 2]
    def __len__(self): return self.batches
    def __iter__(self):
        for _ in range(self.batches):
            ids = self.by_clip[random.choice(self.clips)]
            chosen = random.sample(ids, min(self.p, len(ids)))
            batch = []
            for lab in chosen:
                rows = self.by_pid[lab]
                batch += list(np.random.choice(rows, self.k, replace=len(rows) < self.k))
            yield batch

BATCHES_PER_EPOCH = len(train_df) // (P * K)
train_loader = DataLoader(Crops(train_df, train_tf), batch_sampler=PKSampler(train_df, P, K, BATCHES_PER_EPOCH),
                          num_workers=2, pin_memory=True)
print(f"{BATCHES_PER_EPOCH} batches of {P}x{K} per epoch, device {DEVICE}")

## 5 · Evaluation (the tracker's situation) and the baselines
`embed` turns crops into L2-normalised vectors. Then, for every clip of the test:
* **queries**: up to 20 crops per player;
* **gallery**: all the crops of the same clip at least `GAP` frames away from the query, with team-mates and opponents included.

The cosine distance (1 − dot product) ranks the gallery.

We measure two baselines first:
* **ResNet-50 ImageNet**: what the tracker uses today;
* **OSNet ImageNet**: the right architecture, but not yet trained for ReID.

In [ ]:
import torchvision
from torchreid.reid import models as reid_models

@torch.inference_mode()
def embed(model, df, batch=128) -> np.ndarray:
    model.eval()
    loader = DataLoader(Crops(df, test_tf, labels=False), batch_size=batch, num_workers=2)
    out = []
    for x, _ in loader:
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            f = model(x.to(DEVICE))
        out.append(torch.nn.functional.normalize(f.float().flatten(1), dim=1).cpu().numpy())
    return np.concatenate(out)

def retrieval(feats: np.ndarray, df: pd.DataFrame, gap: int = GAP, per_pid: int = 20):
    # rank-1 and mAP, query vs same-clip gallery at least `gap` frames away
    rng = np.random.default_rng(SEED)
    r1, aps, d_same, d_other = [], [], [], []
    for clip, g in df.groupby("clip"):
        idx = g.index.to_numpy(); f = feats[idx]
        pid, frame = g.pid.to_numpy(), g.frame.to_numpy()
        dist = 1 - f @ f.T                                            # cosine distance, clip x clip
        for p in np.unique(pid):
            qs = np.flatnonzero(pid == p)
            for q in rng.choice(qs, min(per_pid, len(qs)), replace=False):
                valid = np.abs(frame - frame[q]) >= gap
                if not (valid & (pid == p)).any():
                    continue                                          # player not visible 2 s away
                d, same = dist[q, valid], pid[valid] == p
                order = np.argsort(d); hits = same[order]
                r1.append(hits[0])
                prec = np.cumsum(hits) / (np.arange(len(hits)) + 1)
                aps.append((prec * hits).sum() / hits.sum())
                d_same.append(d[same]); d_other.append(d[~same])
    return {"rank1": float(np.mean(r1)), "mAP": float(np.mean(aps)), "queries": len(r1)}, \
           np.concatenate(d_same), np.concatenate(d_other)

def resnet50_imagenet():
    m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
    m.fc = torch.nn.Identity()
    return m.to(DEVICE)

def osnet(num_classes=1):
    return reid_models.build_model("osnet_x1_0", num_classes=num_classes, loss="triplet",
                                   pretrained=PRETRAINED).to(DEVICE)

results, dists = {}, {}
for name, build in (("ResNet-50 ImageNet (today)", resnet50_imagenet), ("OSNet ImageNet", osnet)):
    m = build()
    results[name], *dists[name] = retrieval(embed(m, test_df), test_df)
    print(f"{name:28s} {results[name]}")
    del m

## 6 · Training
Per batch:
1. `model(x)` in training mode returns the identity scores (`logits`) **and** the embedding.
2. `loss = cross_entropy(logits) + triplet(embedding)`.
3. Adam updates the weights. Mixed precision (`autocast`) roughly doubles the speed on a T4.

The **batch-hard triplet** is written out below. It is the heart of the method, so it is not hidden in a library call.

**If Colab disconnects:** run sections 1–5 again and then this cell. It loads `last.pth` from Drive and continues from the next epoch.

In [ ]:
import time
import torch.nn.functional as F

def batch_hard_triplet(emb: torch.Tensor, labels: torch.Tensor, margin: float = MARGIN) -> torch.Tensor:
    emb = F.normalize(emb, dim=1)
    dist = 1 - emb @ emb.T                                           # cosine distance, batch x batch
    same = labels[:, None] == labels[None, :]
    hardest_pos = (dist * same).max(1).values                        # farthest crop of the same player
    hardest_neg = (dist + 2.0 * same).min(1).values                  # closest crop of ANOTHER player
    return F.relu(hardest_pos - hardest_neg + margin).mean()

model = osnet(NUM_IDS)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)
sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[int(EPOCHS * 2 / 3)], gamma=0.1)
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
ce = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

start, history = 0, []
ckpt = RUN / "last.pth"
if ckpt.is_file():
    s = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"])
    start, history = s["epoch"] + 1, s["history"]
    print(f"resuming at epoch {start + 1}")

for epoch in range(start, EPOCHS):
    model.train(); t0 = time.time(); tot = {"id": 0.0, "tri": 0.0, "acc": 0.0}
    for x, y in train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            logits, emb = model(x)                                   # training mode: (scores, embedding)
            l_id, l_tri = ce(logits, y), batch_hard_triplet(emb.float(), y)
            loss = l_id + l_tri
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot["id"] += l_id.item(); tot["tri"] += l_tri.item(); tot["acc"] += (logits.argmax(1) == y).float().mean().item()
    sched.step()
    n = len(train_loader)
    history.append({"epoch": epoch + 1, **{k: v / n for k, v in tot.items()}, "lr": opt.param_groups[0]["lr"]})
    print(f"epoch {epoch + 1:2d}/{EPOCHS}  id-loss {tot['id'] / n:.3f}  triplet {tot['tri'] / n:.3f}  "
          f"train-acc {tot['acc'] / n:.2f}  ({time.time() - t0:.0f} s)")
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "epoch": epoch, "history": history}, ckpt)

# the file the tracker loads (torchreid format; the classifier is discarded when loading)
WEIGHTS = RUN / "osnet_x1_0_sportsmot.pth"
torch.save({"state_dict": model.state_dict()}, WEIGHTS)
print("saved", WEIGHTS)

### Training curves
The identity loss and the triplet loss should go down, with a step when the learning rate drops. The train accuracy says the model recognises the *training* players. It says nothing yet about new players: that is the test below.

In [ ]:
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
h.plot(x="epoch", y="id", ax=ax[0], title="identity loss", legend=False)
h.plot(x="epoch", y="tri", ax=ax[1], title="triplet loss", legend=False)
h.plot(x="epoch", y="acc", ax=ax[2], title="train accuracy", legend=False)
plt.tight_layout(); plt.savefig(RUN / "training_curves.png", dpi=80); plt.show()

## 7 · Evaluation on the 4 unseen arenas

In [ ]:
name = "OSNet fine-tuned (SportsMOT)"
results[name], *dists[name] = retrieval(embed(model, test_df), test_df)
table = pd.DataFrame(results).T.round(3)
display(table)

### Distances: what the tracker's thresholds see
For each model, the cosine distance between crops of the **same player** (green) and of **different players of the same clip** (red), 2 s or more apart. A good ReID model separates the two humps. Where they cross is roughly where the tracker's thresholds belong: `appearance_gate` (0.35 today) and `memory_thresh` (0.25 today) were chosen for ResNet-50. With a new model they must be recalibrated, otherwise a better model can look worse in the tracker.

In [ ]:
fig, axes = plt.subplots(1, len(dists), figsize=(6 * len(dists), 3.5), sharey=True)
calib = {}
for ax, (name, (same, other)) in zip(np.atleast_1d(axes), dists.items()):
    bins = np.linspace(0, max(1.0, float(np.percentile(other, 99))), 60)
    ax.hist(same, bins, alpha=0.6, density=True, color="tab:green", label="same player")
    ax.hist(other, bins, alpha=0.6, density=True, color="tab:red", label="other player, same clip")
    # threshold with equal error: share of same-player pairs above it = share of other pairs below it
    ts = np.linspace(0, bins[-1], 400)
    eer_t = ts[np.argmin(np.abs([(same > t).mean() - (other <= t).mean() for t in ts]))]
    calib[name] = {"same_median": float(np.median(same)), "other_median": float(np.median(other)),
                   "equal_error_threshold": float(eer_t),
                   "other_5th_percentile": float(np.percentile(other, 5))}
    ax.axvline(eer_t, color="k", ls="--", lw=1)
    ax.set_title(f"{name}\nequal-error threshold {eer_t:.2f}", fontsize=10); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(RUN / "distances.png", dpi=80); plt.show()
display(pd.DataFrame(calib).T.round(3))

### Retrieval examples
A query (left) and the 6 closest gallery crops of the same clip, 2 s or more away. Green = same player, red = a different player. Top row: today's ResNet. Bottom row: the fine-tuned OSNet.

In [ ]:
base = resnet50_imagenet()
feats = {"ResNet-50": embed(base, test_df), "OSNet fine-tuned": embed(model, test_df)}
del base
rng = np.random.default_rng(1)
queries = rng.choice(len(test_df), 4, replace=False)
fig, axes = plt.subplots(2 * len(queries), 7, figsize=(11, 4.2 * len(queries)))
for qi, q in enumerate(queries):
    g = test_df[(test_df["clip"] == test_df["clip"][q]) & ((test_df.frame - test_df.frame[q]).abs() >= GAP)]
    for r, (name, f) in enumerate(feats.items()):
        row = axes[2 * qi + r]
        d = 1 - f[g.index] @ f[q]
        row[0].imshow(cv2.imread(test_df.path[q])[..., ::-1]); row[0].set_title(f"query\n{name}", fontsize=7)
        for c, j in enumerate(g.index[np.argsort(d)[:6]], 1):
            img = cv2.imread(test_df.path[j])
            ok = test_df.pid[j] == test_df.pid[q]
            img = cv2.copyMakeBorder(img, 6, 6, 6, 6, cv2.BORDER_CONSTANT, value=(0, 200, 0) if ok else (0, 0, 220))
            row[c].imshow(img[..., ::-1]); row[c].set_title(f"{1 - (f[j] @ f[q]):.2f}", fontsize=7)
        for a in row: a.axis("off")
plt.tight_layout(); plt.savefig(RUN / "retrieval_examples.jpg", dpi=70); plt.show()

## 8 · Save and take it home
1. Download `MyDrive/hooptrack/runs/reid_osnet_x1_0/osnet_x1_0_sportsmot.pth` and save it as `weights/osnet_x1_0_sportsmot.pth`. Also download `eval_012.json` and the figures for `experiments/012_reid_osnet/`.
2. On the laptop: `pip install torchreid gdown`.
3. Measure it in the tracker with the best configuration (011b), changing only the ReID model. The thresholds can be adjusted with `--set` using the calibration table above:

```powershell
python scripts/run_court_source_eval.py --config configs/player.yaml `
    --set reid.type=osnet reid.weights=weights/osnet_x1_0_sportsmot.pth `
    --out outputs/runs/012_reid_osnet --variants segmentation `
    --clips v_00HRwkvvjtQ_c005 v_5ekaksddqrc_c002 v_BgwzTUxJaeU_c012 v_4r8QL_wglzQ_c001
```

In [ ]:
import json
report = {"recipe": {"epochs": EPOCHS, "lr": LR, "P": P, "K": K, "margin": MARGIN, "label_smoothing": LABEL_SMOOTH,
                     "pretrained_imagenet": PRETRAINED, "test_gap_frames": GAP},
          "data": {"train_crops": len(train_df), "train_ids": NUM_IDS, "test_crops": len(test_df),
                   "test_ids": int(test_df.pid.nunique())},
          "retrieval_test": table.to_dict(orient="index"),
          "distance_calibration": calib}
(RUN / "eval_012.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))